In [3]:
import pandas as pd

# Load all sheets into a dictionary of DataFrames
all_sheets = pd.read_excel('Feedipedia_BigQuery_Full_Export.xlsx', sheet_name=None)

# Access a specific sheet by its tab name
df_geo = all_sheets['datasheet_geo']

regions = df_geo["region_code"].unique().tolist()


In [6]:
df_geo.head()

,datasheet_id,datasheet_title,status,source_level,region_code,country_id,country_name,iso3,note
0,468,Cocoa (Theobroma cacao) beans and by-products,native,region,south_america,6.0,Argentina,ARG,Expanded from region code
1,468,Cocoa (Theobroma cacao) beans and by-products,native,region,south_america,23.0,Bolivia (Plurinational State of),BOL,Expanded from region code
2,468,Cocoa (Theobroma cacao) beans and by-products,native,region,south_america,24.0,Brazil,BRA,Expanded from region code
3,468,Cocoa (Theobroma cacao) beans and by-products,native,region,south_america,32.0,Chile,CHL,Expanded from region code
4,468,Cocoa (Theobroma cacao) beans and by-products,native,region,south_america,38.0,Colombia,COL,Expanded from region code


In [11]:
grouped_sets.head()

,region_code,datasheet_id,iso3
0,caribbean,500,"(DMA, HTI, CUB, BRB, ATG, KNA, JAM, BHS, LCA, ..."
1,caribbean,541,"(DMA, HTI, CUB, BRB, ATG, KNA, JAM, BHS, LCA, ..."
2,caribbean,554,"(DMA, HTI, CUB, BRB, ATG, KNA, JAM, BHS, LCA, ..."
3,caribbean,608,"(DMA, HTI, CUB, BRB, ATG, KNA, JAM, BHS, LCA, ..."
4,caribbean,613,"(DMA, HTI, CUB, BRB, ATG, KNA, JAM, BHS, LCA, ..."


In [13]:
grouped_sets.head()

,region_code,datasheet_id,country_set
0,caribbean,500,"((8.0, Antigua And Barbuda, ATG), (70.0, Grena..."
1,caribbean,541,"((8.0, Antigua And Barbuda, ATG), (70.0, Grena..."
2,caribbean,554,"((8.0, Antigua And Barbuda, ATG), (70.0, Grena..."
3,caribbean,608,"((8.0, Antigua And Barbuda, ATG), (70.0, Grena..."
4,caribbean,613,"((8.0, Antigua And Barbuda, ATG), (70.0, Grena..."


In [12]:
df_unique_countries = df_geo[
    ["region_code", "datasheet_id", "country_id", "country_name", "iso3"]
].drop_duplicates()

# Create a frozenset of country metadata tuples for each region and datasheet_id group
grouped_sets = (
    df_unique_countries.groupby(["region_code", "datasheet_id"])
    .apply(
        lambda group: frozenset(
            zip(group["country_id"], group["country_name"], group["iso3"])
        ),
        include_groups=False,
    )
    .reset_index(name="country_set")
)

# ---------------------------------------------------------
# Step 2: Verify Consistency Across datasheet_ids
# ---------------------------------------------------------
consistency_check = (
    grouped_sets.groupby("region_code")["country_set"]
    .nunique()
    .reset_index(name="unique_country_sets_count")
)

is_consistent = (consistency_check["unique_country_sets_count"] == 1).all()

print("--- Region Consistency Analysis ---")
print(consistency_check)

if is_consistent:
    print(
        "\n✅ Success: All datasheet_ids have consistent country sets per region."
    )
else:
    print(
        "\n⚠️ Warning: Inconsistency found! Some datasheet_ids have different country sets for the same region."
    )

# ---------------------------------------------------------
# Step 3: Extract and Save Unique Country Data Sets Per Region
# ---------------------------------------------------------
# Get unique country records across all datasheet_ids per region
region_country_data = (
    df_geo[["region_code", "country_id", "country_name", "iso3"]]
    .drop_duplicates()
    .sort_values(by=["region_code", "country_id"])
)

# Option A: Save as a structured DataFrame to CSV
region_country_data.to_csv("region_countries_master.csv", index=False)

# Option B: Store as a nested Dictionary in Python
region_country_dict = (
    region_country_data.groupby("region_code")
    .apply(
        lambda g: g[["country_id", "country_name", "iso3"]].to_dict(
            orient="records"
        ),
        include_groups=False,
    )
    .to_dict()
)

print("\n--- Consolidated Master Country Records Per Region ---")
print(region_country_data)

--- Region Consistency Analysis ---
              region_code  unique_country_sets_count
0               caribbean                          1
1          central_africa                          1
2         central_america                          1
3            central_asia                          1
4             east_africa                          1
5               east_asia                          1
6                  europe                          1
7           mediterranean                          1
8            north_africa                          1
9           north_america                          1
10                oceania                          1
11          south_america                          1
12             south_asia                          1
13         southeast_asia                          1
14        southern_africa                          1
15            west_africa                          1
16  west_asia_middle_east                          1

✅ Success

In [ ]:
region_country_dict

{'caribbean': [{'country_id': 8.0,
   'country_name': 'Antigua And Barbuda',
   'iso3': 'ATG'},
  {'country_id': 19.0, 'country_name': 'Bahamas', 'iso3': 'BHS'},
  {'country_id': 25.0, 'country_name': 'Barbados', 'iso3': 'BRB'},
  {'country_id': 42.0, 'country_name': 'Cuba', 'iso3': 'CUB'},
  {'country_id': 47.0, 'country_name': 'Dominica', 'iso3': 'DMA'},
  {'country_id': 49.0, 'country_name': 'Dominican Republic', 'iso3': 'DOM'},
  {'country_id': 70.0, 'country_name': 'Grenada', 'iso3': 'GRD'},
  {'country_id': 76.0, 'country_name': 'Haiti', 'iso3': 'HTI'},
  {'country_id': 86.0, 'country_name': 'Jamaica', 'iso3': 'JAM'},
  {'country_id': 94.0, 'country_name': 'Saint Kitts And Nevis', 'iso3': 'KNA'},
  {'country_id': 101.0, 'country_name': 'Saint Lucia', 'iso3': 'LCA'},
  {'country_id': 179.0, 'country_name': 'Trinidad And Tobago', 'iso3': 'TTO'},
  {'country_id': 190.0,
   'country_name': 'Saint Vincent and the Grenadines',
   'iso3': 'VCT'}],
 'central_africa': [{'country_id': 29.0

: 